In [1]:
import polars as pl
from polars import col
from investment_strategy.data.cleaner import *
from investment_strategy.signals.signal_construction import *
from investment_strategy.signals.signal_ranking import *
from investment_strategy.portfolio.weighting import *
from investment_strategy.portfolio.rebalancing import *
from investment_strategy.portfolio.valuation import *
from investment_strategy.analytics.return_metrics import *
from investment_strategy.analytics.risk_metrics import *
from investment_strategy.config.backtest_config import *
from datetime import date

market_data = pl.read_parquet("../data/raw/sp500_market_data.parquet")
market_data

date,ticker,open,high,low,close,volume
date,str,f64,f64,f64,f64,i64
2018-12-31,"""A""",62.795802,63.874904,62.795802,63.855968,1572100
2018-12-31,"""AAPL""",37.613949,37.810881,37.127551,37.42651,140014000
2018-12-31,"""ABBV""",66.040205,67.042343,65.773452,66.465576,5722100
2018-12-31,"""ABNB""",null,null,null,null,null
2018-12-31,"""ABT""",62.168729,63.202416,62.090555,62.828899,6094300
…,…,…,…,…,…,…
2026-05-29,"""XYZ""",74.970001,76.660004,74.195,75.720001,7380400
2026-05-29,"""YUM""",149.229996,150.179993,147.339996,147.949997,3992700
2026-05-29,"""ZBH""",81.952229,82.979498,81.363796,82.111809,3216600


# Signal Construction

In [2]:
market_data = fill_OHLCV_missing_values(market_data)
close_price = market_data.select(
    col("date"),
    col("ticker"),
    col("close")
)

In [3]:
end_date = get_backtest_end_date(
    backtest_start_date=BACKTEST_START_DATE,
    backtest_period=BACKTEST_PERIOD,
    backtest_period_unit=BACKTEST_PERIOD_UNIT,
)
end_date

datetime.date(2024, 12, 31)

In [4]:
trading_calendar = get_trading_calendar(cleaned_close_prices_dataset=close_price)
trading_calendar

date
date
2018-12-31
2019-01-02
2019-01-03
2019-01-04
2019-01-07
…
2026-05-22
2026-05-26
2026-05-27


In [5]:
date_mapping_df = create_date_mapping(
    trading_calendar=trading_calendar,
    rebalance_frequency=REBALANCE_FREQUENCY,
    rebalance_freq_unit=REBALANCE_FREQ_UNIT,
    backtest_start_date=BACKTEST_START_DATE,
    backtest_end_date=end_date,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
date_mapping_df

lookback_date,lag_base_date,signal_date,rebalance_date
date,date,date,date
2021-06-30,2021-11-30,2021-12-30,2021-12-31
2021-08-25,2022-01-25,2022-02-25,2022-02-28
2021-10-29,2022-03-29,2022-04-29,2022-05-02
2021-12-29,2022-05-27,2022-06-29,2022-06-30
2022-02-28,2022-07-29,2022-08-30,2022-08-31
…,…,…,…
2023-10-27,2024-03-28,2024-04-29,2024-04-30
2023-12-28,2024-05-28,2024-06-28,2024-07-01
2024-02-29,2024-07-30,2024-08-30,2024-09-03


In [6]:
price_date_df = get_prices_for_date_mapping(
    cleaned_close_prices_dataset=close_price,
    cleaned_stock_OHLCV=market_data,
    date_mapping_df=date_mapping_df,
)
price_date_df

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open
date,str,f64,date,date,date,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822
…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847


In [7]:
past_returns = calculate_momentum(
    factor_reference_table=price_date_df,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
past_returns

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo
date,str,f64,date,date,date,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214
…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519


In [8]:
past_std = calculate_past_returns_std(
    cleaned_close_prices_dataset=close_price,
    factor_reference_table=past_returns,
    date_mapping_df=date_mapping_df,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
past_std

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo
date,str,f64,date,date,date,f64,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467
…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842


In [9]:
risk_adjusted_table = get_risk_adjusted_return(
    factor_reference_with_momentum_and_vol=past_std,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
risk_adjusted_table

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249,1.883082
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089,16.081247
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661,3.743267
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615,4.587432
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467,8.905733
…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146,13.743568
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647,5.086498
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842,2.221282


# Signal Ranking

In [10]:
signal_ranked = rank_signal(
    signal_df=risk_adjusted_table,
    signal_col=f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT}",
)
signal_ranked

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249,1.883082,255
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089,16.081247,42
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661,3.743267,213
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615,4.587432,195
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467,8.905733,111
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146,13.743568,191
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647,5.086498,341
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842,2.221282,376


In [11]:
filtered_ticker = filter_top_ranked(
    ranked_signal_df=signal_ranked,
    rank_col=f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT} rank",
    top_n=TOP_N,
)
filtered_ticker

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""ALB""",221.557343,2021-06-30,2021-11-30,2021-12-31,158.740082,251.533096,221.235862,0.584559,0.02389,24.468939,10
2021-12-30,"""AMD""",145.149994,2021-06-30,2021-11-30,2021-12-31,93.93,158.369995,146.160004,0.686043,0.025971,26.415665,8
2021-12-30,"""BLDR""",84.050003,2021-06-30,2021-11-30,2021-12-31,42.66,69.440002,84.290001,0.627754,0.020418,30.7454,3
2021-12-30,"""COST""",535.266479,2021-06-30,2021-11-30,2021-12-31,374.264069,511.982544,534.791945,0.367971,0.010224,35.990245,2
2021-12-30,"""DDOG""",178.929993,2021-06-30,2021-11-30,2021-12-31,104.080002,178.289993,179.190002,0.713009,0.026257,27.15499,6
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""NI""",35.226597,2024-06-28,2024-11-29,2024-12-31,27.213289,36.560795,35.274594,0.34349,0.009243,37.160354,5
2024-12-30,"""PLTR""",77.18,2024-06-28,2024-11-29,2024-12-31,25.33,67.080002,77.580002,1.648243,0.037635,43.795033,3
2024-12-30,"""TPL""",365.909271,2024-06-28,2024-11-29,2024-12-31,238.746994,528.161865,367.419189,1.212224,0.026436,45.855291,2


In [12]:
sorted_ranking = sort_rankings(
    filtered_signal_df=filtered_ticker,
    rank_col=f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT} rank",
)
sorted_ranking

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""FDS""",461.847412,2021-06-30,2021-11-30,2021-12-31,318.493347,446.441071,461.847424,0.401728,0.010503,38.250071,1
2021-12-30,"""COST""",535.266479,2021-06-30,2021-11-30,2021-12-31,374.264069,511.982544,534.791945,0.367971,0.010224,35.990245,2
2021-12-30,"""BLDR""",84.050003,2021-06-30,2021-11-30,2021-12-31,42.66,69.440002,84.290001,0.627754,0.020418,30.7454,3
2021-12-30,"""VRSK""",221.089966,2021-06-30,2021-11-30,2021-12-31,168.90416,217.692719,220.925249,0.288854,0.009398,30.735613,4
2021-12-30,"""ODFL""",173.97641,2021-06-30,2021-11-30,2021-12-31,123.783798,173.439255,173.854314,0.401147,0.013245,30.28613,5
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""TRGP""",172.180573,2024-06-28,2024-11-29,2024-12-31,123.482193,197.887573,172.538972,0.60256,0.016254,37.070911,6
2024-12-30,"""ATO""",134.360123,2024-06-28,2024-11-29,2024-12-31,111.464882,146.342606,134.872702,0.312903,0.008631,36.252237,7
2024-12-30,"""AXON""",604.320007,2024-06-28,2024-11-29,2024-12-31,294.23999,646.960022,607.169983,1.198749,0.033564,35.715292,8


# Portfolio Weights

In [13]:
equal_weighted_portfolio = construct_portfolio_weights(
    filtered_signal_df=filtered_ticker, weighting_method="equal_weighted"
)
equal_weighted_portfolio

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank,portfolio_weight
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32,f64
2021-12-30,"""ALB""",221.557343,2021-06-30,2021-11-30,2021-12-31,158.740082,251.533096,221.235862,0.584559,0.02389,24.468939,10,0.1
2021-12-30,"""AMD""",145.149994,2021-06-30,2021-11-30,2021-12-31,93.93,158.369995,146.160004,0.686043,0.025971,26.415665,8,0.1
2021-12-30,"""BLDR""",84.050003,2021-06-30,2021-11-30,2021-12-31,42.66,69.440002,84.290001,0.627754,0.020418,30.7454,3,0.1
2021-12-30,"""COST""",535.266479,2021-06-30,2021-11-30,2021-12-31,374.264069,511.982544,534.791945,0.367971,0.010224,35.990245,2,0.1
2021-12-30,"""DDOG""",178.929993,2021-06-30,2021-11-30,2021-12-31,104.080002,178.289993,179.190002,0.713009,0.026257,27.15499,6,0.1
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""NI""",35.226597,2024-06-28,2024-11-29,2024-12-31,27.213289,36.560795,35.274594,0.34349,0.009243,37.160354,5,0.1
2024-12-30,"""PLTR""",77.18,2024-06-28,2024-11-29,2024-12-31,25.33,67.080002,77.580002,1.648243,0.037635,43.795033,3,0.1
2024-12-30,"""TPL""",365.909271,2024-06-28,2024-11-29,2024-12-31,238.746994,528.161865,367.419189,1.212224,0.026436,45.855291,2,0.1


In [14]:
rebalance_allocation_df = prepare_rebalance_allocation_df(
    weighted_portfolio_signal_df=equal_weighted_portfolio
)
rebalance_allocation_df

rebalance_date,ticker,rebalance_open,portfolio_weight
date,str,f64,f64
2021-12-31,"""ALB""",221.235862,0.1
2021-12-31,"""AMD""",146.160004,0.1
2021-12-31,"""BLDR""",84.290001,0.1
2021-12-31,"""COST""",534.791945,0.1
2021-12-31,"""DDOG""",179.190002,0.1
…,…,…,…
2024-12-31,"""NI""",35.274594,0.1
2024-12-31,"""PLTR""",77.580002,0.1
2024-12-31,"""TPL""",367.419189,0.1


In [15]:
rebalance_date = date_mapping_df.get_column("rebalance_date")
rebalance_date

rebalance_date
date
2021-12-31
2022-02-28
2022-05-02
2022-06-30
2022-08-31
…
2024-04-30
2024-07-01
2024-09-03


# Portfolio Construction

In [16]:
rebalance_summary = run_rebalance_simulation(
    factor_reference_table=price_date_df,
    rebalance_allocation_df=rebalance_allocation_df,
    initial_capital=INITIAL_CAPITAL,
    rebalance_dates=rebalance_date,
)
rebalance_summary

{'rebalance_level_table': shape: (19, 3)
 ┌────────────────┬─────────────────┬───────────────┐
 │ rebalance_date ┆ portfolio_value ┆ cash_residual │
 │ ---            ┆ ---             ┆ ---           │
 │ date           ┆ f64             ┆ f64           │
 ╞════════════════╪═════════════════╪═══════════════╡
 │ 2021-12-31     ┆ 1e6             ┆ 1209.597879   │
 │ 2022-02-28     ┆ 857104.203652   ┆ 298.747987    │
 │ 2022-05-02     ┆ 930069.019201   ┆ 587.403802    │
 │ 2022-06-30     ┆ 884926.691164   ┆ 366.513326    │
 │ 2022-08-31     ┆ 1.0059e6        ┆ 757.500926    │
 │ …              ┆ …               ┆ …             │
 │ 2024-04-30     ┆ 1.8761e6        ┆ 427.092887    │
 │ 2024-07-01     ┆ 1.9479e6        ┆ 844.564733    │
 │ 2024-09-03     ┆ 1.9671e6        ┆ 614.809403    │
 │ 2024-10-31     ┆ 2.0510e6        ┆ 1847.567622   │
 │ 2024-12-31     ┆ 2.0035e6        ┆ 1245.364121   │
 └────────────────┴─────────────────┴───────────────┘,
 'position_level_table': shape: (190, 3)

# Portfolio Valuation

In [17]:
rebalance_period_close_prices = get_backtest_period_close_prices(
    cleaned_close_prices_dataset=close_price,
    backtest_start_date=BACKTEST_START_DATE,
    backtest_end_date=end_date,
)
rebalance_period_close_prices

date,ticker,close
date,str,f64
2021-12-31,"""A""",154.27504
2021-12-31,"""AAPL""",173.599014
2021-12-31,"""ABBV""",114.065155
2021-12-31,"""ABNB""",166.490005
2021-12-31,"""ABT""",128.19101
…,…,…
2024-12-31,"""XYZ""",84.989998
2024-12-31,"""YUM""",130.370239
2024-12-31,"""ZBH""",104.037064


In [18]:
next_date_matched_rebalance_level_table = get_next_date_matched_rebalance_level_table(
    rebalance_level_table=rebalance_summary["rebalance_level_table"]
)
next_date_matched_rebalance_level_table

rebalance_date,portfolio_value,cash_residual,next_rebalance_date
date,f64,f64,date
2021-12-31,1e6,1209.597879,2022-02-28
2022-02-28,857104.203652,298.747987,2022-05-02
2022-05-02,930069.019201,587.403802,2022-06-30
2022-06-30,884926.691164,366.513326,2022-08-31
2022-08-31,1.0059e6,757.500926,2022-10-31
…,…,…,…
2024-04-30,1.8761e6,427.092887,2024-07-01
2024-07-01,1.9479e6,844.564733,2024-09-03
2024-09-03,1.9671e6,614.809403,2024-10-31


In [19]:
daily_position_value_table = get_daily_position_value_table(
    backtest_period_close_prices=rebalance_period_close_prices,
    next_date_matched_rebalance_level_table=next_date_matched_rebalance_level_table,
    position_level_table=rebalance_summary["position_level_table"],
)
daily_position_value_table

date,rebalance_date,ticker,shares,close,position_value
date,date,str,i64,f64,f64
2021-12-31,2021-12-31,"""ALB""",452,221.008972,99896.05542
2021-12-31,2021-12-31,"""AMD""",684,143.899994,98427.595825
2021-12-31,2021-12-31,"""BLDR""",1186,85.709999,101652.058914
2021-12-31,2021-12-31,"""COST""",186,538.864075,100228.717896
2021-12-31,2021-12-31,"""DDOG""",558,178.110001,99385.380341
…,…,…,…,…,…
2024-12-31,2024-12-31,"""NI""",5679,35.284191,200378.921436
2024-12-31,2024-12-31,"""PLTR""",2582,75.629997,195276.652908
2024-12-31,2024-12-31,"""TPL""",545,365.423492,199155.803375


In [20]:
daily_portfolio_table = get_daily_portfolio_table(
    next_date_matched_rebalance_level_table=next_date_matched_rebalance_level_table,
    daily_position_value_table=daily_position_value_table,
    initial_capital=INITIAL_CAPITAL
)
daily_portfolio_table

date,positions_value,cash_residual,portfolio_value,daily_return
date,f64,f64,f64,f64
2021-12-31,1.0007e6,1209.597879,1.0019e6,0.001936
2022-01-03,985251.542358,1209.597879,986461.140237,-0.015445
2022-01-04,980925.477905,1209.597879,982135.075784,-0.004385
2022-01-05,939963.130768,1209.597879,941172.728647,-0.041707
2022-01-06,937454.747757,1209.597879,938664.345636,-0.002665
…,…,…,…,…
2024-12-24,2.0320e6,1847.567622,2.0339e6,0.006457
2024-12-26,2.0281e6,1847.567622,2.0300e6,-0.001914
2024-12-27,2.0118e6,1847.567622,2.0137e6,-0.008034


# Analytics

In [21]:
daily_portfolio_value_return_df = prepare_daily_portfolio_value_return_df(
    daily_portfolio_table=daily_portfolio_table
)
daily_portfolio_value_return_df

date,portfolio_value,daily_return
date,f64,f64
2021-12-31,1.0019e6,0.001936
2022-01-03,986461.140237,-0.015445
2022-01-04,982135.075784,-0.004385
2022-01-05,941172.728647,-0.041707
2022-01-06,938664.345636,-0.002665
…,…,…
2024-12-24,2.0339e6,0.006457
2024-12-26,2.0300e6,-0.001914
2024-12-27,2.0137e6,-0.008034


In [22]:
total_return = calculate_total_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    initial_capital=INITIAL_CAPITAL
)
total_return

0.9855164783821788

In [23]:
annualized_return_cagr = calculate_annualized_return_cagr(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    initial_capital=INITIAL_CAPITAL
)
annualized_return_cagr

0.25801768845197426

In [24]:
mean_daily_return = calculate_mean_daily_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
mean_daily_return

0.0010377026988178455

In [25]:
annualized_mean_daily_return = calculate_annualized_mean_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
annualized_mean_daily_return

0.29870224619539876

In [26]:
mean_daily_std = calculate_daily_return_std(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
mean_daily_std

0.015973502482870435

In [27]:
annualized_volatility = calculate_annualized_volatility(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
annualized_volatility

0.2535714908180877

In [28]:
drawdown = calculate_drawdown(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
drawdown

date,portfolio_value,daily_return,drawdown
date,f64,f64,f64
2021-12-31,1.0019e6,0.001936,0.0
2022-01-03,986461.140237,-0.015445,-0.015445
2022-01-04,982135.075784,-0.004385,-0.019763
2022-01-05,941172.728647,-0.041707,-0.060646
2022-01-06,938664.345636,-0.002665,-0.06315
…,…,…,…
2024-12-24,2.0339e6,0.006457,-0.070722
2024-12-26,2.0300e6,-0.001914,-0.0725
2024-12-27,2.0137e6,-0.008034,-0.079952


In [29]:
max_drawdown = calculate_max_drawdown(drawdown_table=drawdown)
max_drawdown

-0.20132831850389232

In [30]:
sharpe_ratio = calculate_sharpe_ratio(
    mean_daily_return=mean_daily_return, annualized_volatility=annualized_volatility
)
sharpe_ratio

1.0312716120350376

In [31]:
calmer_ratio = calculate_calmer_ratio(
    annualized_return_cagr=annualized_return_cagr, max_drawdown=max_drawdown
)
calmer_ratio

1.2815767318246685